In [ ]:

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()
True
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
class SentimentSchema(BaseModel):
    sentiment: Literal['postive','negative'] = Field(description='Sentiment of the review')

    # In Python and Agentic AI, Literal is used when you want a variable or a model field to accept only specific, predefined values instead of any value.

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)
prompt = 'What is the sentiment of the follwing review - The software too bad'
structured_model.invoke(prompt).sentiment
structured_model2.invoke(prompt)
DiagnosisSchema(issue_type='Other', tone='disappointed', urgency='low')
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str
def find_sentiment(state : ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}
def positive_response(state: ReviewState):
    
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
    """
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()} 
def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
    response = model.invoke(prompt).content

    return {'response': response}
def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
graph = StateGraph(ReviewState)

# add nodes
graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

# add edges with conditions
graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)

graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()
workflow

intial_state = {
    'review': "software with very poor infrastructure and desigining is so poor i event cannot handle important functions properly, so poorly styled website and very complicated to use"
}
result = workflow.invoke(intial_state)
result
{'review': 'software with very poor infrastructure and desigining is so poor i event cannot handle important functions properly, so poorly styled website and very complicated to use',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'UX', 'tone': 'frustrated', 'urgency': 'high'},
 'response': 'Subject: Resolution: Your Urgent UX Issue - [Specific Feature/Area if known]\n\nHi [User Name],\n\nI\'m truly sorry to hear you\'ve been experiencing a frustrating UX issue, and I completely understand why you\'ve marked this with high urgency. It\'s incredibly disruptive when things don\'t work as expected, and we appreciate you bringing this to our attention so promptly.\n\nOur team immediately prioritized your report and after investigating, we\'ve identified the root cause of the [briefly mention the specific UX issue if known, e.g., \'lagging interface,\' \'broken button,\' \'confusing workflow\'].\n\n**Here\'s what we\'ve done to resolve it:**\n\n*   **[Specific Action Taken, e.g., "We\'ve deployed a hotfix that addresses the rendering issue on the dashboard.", "We\'ve rolled back the recent update that was causing the button malfunction.", "We\'ve adjusted the workflow logic to make the navigation clearer."].**\n*   **[If applicable, add a user-side action, e.g., "To ensure the changes take full effect, please clear your browser cache and cookies, then refresh the page/restart the application."].**\n\nThe issue should now be resolved, and you should find the experience much smoother and more intuitive.\n\nCould you please take a moment to check on your end and confirm that everything is working as expected now? Your feedback is crucial to ensure we\'ve fully addressed your concerns.\n\nWe\'re also taking steps to review our [relevant process, e.g., testing protocols, deployment procedures] to prevent similar UX issues from occurring in the future. Your experience helps us improve for everyone.\n\nPlease don\'t hesitate to reach out immediately if you encounter any further difficulties or if the issue persists. We\'re here to ensure you have a seamless experience.\n\nThanks again for your patience and understanding.\n\nBest regards,\n\n[Your Name]\nSupport Assistant'}
 